
# FibroLight-VLNet — Full Working Colab Implementation

**Based on:** *Explainable Optimization-Driven Multimodal Vision–Language Learning for Ovarian Thecoma–Fibroma Tumor Detection Using Ultrasound Images and Clinical Narratives*

This notebook implements the paper-aligned pipeline: ultrasound preprocessing, diagnosis-masked clinical narratives, lightweight Swin Transformer, truncated Bio_ClinicalBERT, Cross-Modal Bidirectional Attention Fusion (CBAF), LoRA-efficient training, Adaptive Differential Evolution Optimization (ADEO), gated Mixture-of-Experts classification, leakage controls, external narrative-free specificity evaluation, and FMEE-style multimodal attribution/counterfactual explanation.

The paper constructs **class-conditional pseudo-pairs**, because it states that no public dataset provides paired ultrasound and clinical narratives for the same ovarian patient. The positive class is the **theca-cell-tumor / thecoma–fibroma group**. For real experiments, configure the dataset paths in Cell 2. A synthetic demo mode is included so the notebook can be tested end-to-end.


In [ ]:

# ============================================================
# 1. INSTALL + IMPORTS
# ============================================================
!pip -q install -U timm transformers accelerate sentencepiece opencv-python-headless scikit-learn shap openpyxl scipy pandas matplotlib seaborn pillow

import os, re, gc, json, math, random, time, hashlib, shutil, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, balanced_accuracy_score,
    matthews_corrcoef, roc_curve, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


In [ ]:

# ============================================================
# 2. GOOGLE DRIVE + CONFIGURATION
# ============================================================
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# --------- EDIT THESE FOR REAL DATA ---------
MMOTU_ROOT = "/content/drive/MyDrive/MMOTU"
PMC_PATIENTS_FILE = "/content/drive/MyDrive/PMC-Patients.csv"
EXTERNAL_ROOT = "/content/drive/MyDrive/Ovarian_External"
MMOTU_METADATA = ""

# --------- RUN MODE ---------
DEMO_MODE = True
REAL_DATA_MODE = False
FAST_MODE = True

RUN_ADEO = True
RUN_EXPLANATIONS = True

OUTPUT_DIR = "/content/drive/MyDrive/FibroLight_VLNet_Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Paper-aligned settings
IMAGE_SIZE = 224
MAX_TEXT_LEN = 256
MASK_PROB = 0.30
CBAF_DIM = 128
AFFINITY_DIM = 32
FUSED_DIM = 256
VISUAL_TOKENS = 49
LR = 2e-5
WEIGHT_DECAY = 0.01
FOCAL_GAMMA = 2.0
BALANCE_WEIGHT = 0.01
GRAD_CLIP = 1.0
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
EXPERT_MIN, EXPERT_MAX = 2, 8
WIDTH_MIN, WIDTH_MAX = 8, 64

# FAST_MODE makes the notebook practical on Colab.
# False uses the paper's larger settings.
if FAST_MODE:
    EPOCHS, BATCH_SIZE = 3, 4
    ADEO_POP, ADEO_GEN, ADEO_INNER_FOLDS = 8, 3, 2
    EXPLAIN_PERMUTATIONS = 8
else:
    EPOCHS, BATCH_SIZE = 60, 32
    ADEO_POP, ADEO_GEN, ADEO_INNER_FOLDS = 60, 50, 3
    EXPLAIN_PERMUTATIONS = 64

print("FAST_MODE:", FAST_MODE)
print("Epochs:", EPOCHS, "Batch:", BATCH_SIZE)
print("ADEO:", ADEO_POP, "population x", ADEO_GEN, "generations")



## Dataset construction

The paper uses MMOTU OTU_2D as the primary ultrasound source and PMC-Patients as the narrative source. MMOTU contains 1,469 images from 247 patients, with the theca-cell-tumor class used as the positive thecoma–fibroma group. Narratives are retrieved by ovarian-mass terminology and paired at the class level. This notebook follows that design rather than pretending the sources are patient-matched.


In [ ]:

# ============================================================
# 3. DEMO DATA + REAL-DATA LOADERS
# ============================================================
def make_demo_dataset(root="/content/fibro_demo", n=160):
    root = Path(root); img_root = root/"images"; img_root.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(SEED); rows = []
    for i in range(n):
        y = int(i < n//4)
        patient = f"P{i:04d}"
        a = rng.normal(80 if not y else 95, 25, (224,224)).clip(0,255).astype(np.uint8)
        a = cv2.GaussianBlur(a,(5,5),0)
        if y:
            yy,xx=np.ogrid[:224,:224]; cx,cy=rng.integers(70,155,2); r=rng.integers(25,55)
            lesion=((xx-cx)**2+(yy-cy)**2)<r*r
            a[lesion]=np.clip(a[lesion]+rng.integers(35,70),0,255)
        p=img_root/f"{patient}.png"; Image.fromarray(a).save(p)
        txt=("Patient presents with an adnexal mass. Ultrasound demonstrates a predominantly "
             "solid lesion with heterogeneous echogenicity. The examination shows preserved "
             "ovarian architecture and no obvious ascites." if y else
             "Patient presents for evaluation of an adnexal lesion. Ultrasound demonstrates a "
             "cystic or predominantly fluid-filled lesion. The examination shows no obvious "
             "ascites and routine laboratory findings are unremarkable.")
        rows.append([patient,str(p),"theca_cell_tumor" if y else "other_ovarian_lesion",y,txt])
    return pd.DataFrame(rows,columns=["patient_id","image_path","category","label","narrative"])

def find_images(root):
    root=Path(root)
    out=[]
    for e in ("*.png","*.jpg","*.jpeg","*.bmp","*.tif","*.tiff","*.webp"):
        out += list(root.rglob(e))
    return sorted(map(str,out))

def load_real_mmotu(root, metadata_path=""):
    root=Path(root)
    meta=None
    if metadata_path and Path(metadata_path).exists():
        meta=pd.read_csv(metadata_path)
    else:
        csvs=list(root.rglob("*.csv"))
        if csvs:
            try: meta=pd.read_csv(csvs[0])
            except: pass
    if meta is not None:
        lc={c.lower():c for c in meta.columns}
        ic=next((lc[x] for x in ["image_path","image","path","filename","file"] if x in lc),None)
        yc=next((lc[x] for x in ["label","class","category","diagnosis"] if x in lc),None)
        pc=next((lc[x] for x in ["patient_id","patient","id"] if x in lc),None)
        if ic and yc:
            out=pd.DataFrame()
            out["image_path"]=meta[ic].astype(str).map(lambda x: str(root/x) if not os.path.isabs(x) else x)
            out["category"]=meta[yc].astype(str)
            out["patient_id"]=meta[pc].astype(str) if pc else out["image_path"].map(lambda x:Path(x).stem)
            out["label"]=out["category"].str.lower().str.contains(
                "theca|fibroma|thecoma",regex=True).astype(int)
            return out[out.image_path.map(os.path.exists)].reset_index(drop=True)
    rows=[]
    for p in find_images(root):
        rows.append([Path(p).stem,p,Path(p).parent.name,
                     int(bool(re.search(r"theca|fibroma|thecoma",p.lower())))])
    return pd.DataFrame(rows,columns=["patient_id","image_path","category","label"])

def load_pmc(path):
    if not Path(path).exists():
        return pd.DataFrame(columns=["patient_id","category","narrative"])
    try: df=pd.read_csv(path)
    except: df=pd.read_csv(path,sep="\t")
    lc={c.lower():c for c in df.columns}
    tc=next((lc[x] for x in ["text","abstract","summary","patient_summary","case_summary","description"] if x in lc),None)
    ic=next((lc[x] for x in ["patient_id","patient","id","pmcid","article_id"] if x in lc),None)
    if tc is None: raise ValueError("PMC file needs a narrative/summary text column.")
    out=pd.DataFrame()
    out["patient_id"]=df[ic].astype(str) if ic else np.arange(len(df)).astype(str)
    out["narrative"]=df[tc].fillna("").astype(str)
    out["category"]=infer_category(out.narrative)
    return out

def infer_category(texts):
    def f(t):
        t=t.lower()
        if re.search(r"\b(fibroma|thecoma|theca cell)\b",t): return "theca_cell_tumor"
        if re.search(r"\b(teratoma|dermoid)\b",t): return "teratoma"
        if re.search(r"\b(endometrioma)\b",t): return "endometrioma"
        if re.search(r"\b(cystadenoma)\b",t): return "cystadenoma"
        return "other_ovarian_lesion"
    return texts.map(f)

if DEMO_MODE or not REAL_DATA_MODE:
    data=make_demo_dataset()
    narratives=data[["patient_id","category","narrative"]].copy()
else:
    data=load_real_mmotu(MMOTU_ROOT,MMOTU_METADATA)
    narratives=load_pmc(PMC_PATIENTS_FILE)

print("Images:",len(data)," Positive:",int(data.label.sum()))
display(data.head())


In [ ]:

# ============================================================
# 4. IMAGE PREPROCESSING
# ============================================================
def adaptive_median_filter(img,max_window=7):
    # Exact adaptive median implementation for reproducibility.
    img=np.asarray(img,dtype=np.uint8); out=img.copy()
    pad=max_window//2
    x=cv2.copyMakeBorder(img,pad,pad,pad,pad,cv2.BORDER_REFLECT)
    h,w=img.shape
    for i in range(h):
        for j in range(w):
            z=None
            for ws in range(3,max_window+1,2):
                r=ws//2; q=x[i+pad-r:i+pad+r+1,j+pad-r:j+pad+r+1]
                zmin,zmed,zmax=q.min(),np.median(q),q.max()
                if zmin<zmed<zmax:
                    z=img[i,j] if zmin<img[i,j]<zmax else zmed; break
            out[i,j]=np.uint8(np.clip(z if z is not None else zmed,0,255))
    return out

def preprocess_ultrasound(path,size=224,exact=False):
    img=cv2.imread(str(path),cv2.IMREAD_UNCHANGED)
    if img is None: raise FileNotFoundError(path)
    if img.ndim==3: img=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    img=cv2.normalize(img,None,0,255,cv2.NORM_MINMAX).astype(np.uint8)
    img=adaptive_median_filter(img,7) if exact else cv2.medianBlur(img,5)
    img=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(img)
    img=cv2.resize(img,(size,size),interpolation=cv2.INTER_AREA)
    return np.repeat(img[...,None],3,axis=2)

p=data.iloc[0].image_path
raw=cv2.imread(p,cv2.IMREAD_GRAYSCALE)
pr=preprocess_ultrasound(p)
plt.figure(figsize=(11,4))
plt.subplot(1,2,1); plt.imshow(raw,cmap="gray"); plt.title("Raw ultrasound",fontweight="bold"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(pr[:,:,0],cmap="gray"); plt.title("MinMax + Median + CLAHE",fontweight="bold"); plt.axis("off")
plt.tight_layout(); plt.show()


In [ ]:

# ============================================================
# 5. NARRATIVE FILTERING + DIAGNOSIS MASKING
# ============================================================
PRESENTATION_CUES=["present","complain","symptom","pain","bleeding","menstrual","history",
                   "examination","exam","ultrasound","sonograph","imaging","laboratory","lab",
                   "marker","mass","lesion","adnexal","ovarian"]
EXCLUSION_CUES=["histopath","pathology","diagnosis","diagnosed","operation","operative",
                "surgery","surgical","treatment","therapy","chemotherapy","outcome","follow-up"]
DIAGNOSIS_TERMS=["fibroma","fibromas","thecoma","thecomas","theca-cell tumor","theca cell tumor",
                 "teratoma","dermoid","cystadenoma","serous cystadenoma","mucinous cystadenoma",
                 "endometrioma","ovarian cancer","ovarian carcinoma","malignant","benign",
                 "sex cord stromal tumor"]

def filter_and_mask(text):
    kept=[]
    for s in re.split(r"(?<=[.!?])\s+",str(text)):
        lo=s.lower()
        if any(x in lo for x in EXCLUSION_CUES): continue
        if any(x in lo for x in PRESENTATION_CUES): kept.append(s.strip())
    if not kept: kept=re.split(r"(?<=[.!?])\s+",str(text))[:3]
    out=" ".join(kept)
    for term in DIAGNOSIS_TERMS:
        out=re.sub(r"\b"+re.escape(term)+r"\b","[MASK]",out,flags=re.I)
    return re.sub(r"\s+"," ",out).strip()

if len(narratives):
    pools=narratives.groupby("category")["narrative"].apply(list).to_dict()
else:
    pools={}

def choose_narrative(row):
    pool=pools.get(row.category,[])
    if not pool: return "Ultrasound examination demonstrates an adnexal lesion with clinical findings."
    seed=int(hashlib.md5(str(row.patient_id).encode()).hexdigest()[:8],16)
    return random.Random(seed).choice(pool)

data["narrative"]=data.apply(choose_narrative,axis=1)
data["masked_narrative"]=data.narrative.map(filter_and_mask)
display(data[["category","label","masked_narrative"]].head())


In [ ]:

# ============================================================
# 6. SWIN + TRUNCATED BIO_CLINICALBERT + LoRA
# ============================================================
TEXT_CHECKPOINT="emilyalsentzer/Bio_ClinicalBERT"
tokenizer=AutoTokenizer.from_pretrained(TEXT_CHECKPOINT)

class LoRALinear(nn.Module):
    def __init__(self,base,rank=8,alpha=16,dropout=0.05):
        super().__init__(); self.base=base
        self.scale=alpha/max(rank,1); self.drop=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        return self.base(x)+F.linear(self.drop(x),self.B@self.A)*self.scale

class ClinicalEncoder(nn.Module):
    def __init__(self,layers=2):
        super().__init__()
        self.model=AutoModel.from_pretrained(TEXT_CHECKPOINT)
        if hasattr(self.model,"encoder"):
            self.model.encoder.layer=nn.ModuleList(list(self.model.encoder.layer[:layers]))
        for p in self.model.parameters(): p.requires_grad=False
        for layer in self.model.encoder.layer:
            layer.attention.self.query=LoRALinear(layer.attention.self.query,LORA_RANK,LORA_ALPHA,LORA_DROPOUT)
            layer.attention.self.value=LoRALinear(layer.attention.self.value,LORA_RANK,LORA_ALPHA,LORA_DROPOUT)
        self.hidden=self.model.config.hidden_size
    def forward(self,input_ids,attention_mask):
        return self.model(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state

class VisualEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=timm.create_model("swin_tiny_patch4_window7_224",pretrained=True,num_classes=0)
        self.hidden=self.backbone.num_features
        for p in self.backbone.parameters(): p.requires_grad=False
    def forward(self,x):
        z=self.backbone.forward_features(x)
        if z.ndim==4:
            # timm Swin output is normally B x 7 x 7 x C
            if z.shape[1]==7 and z.shape[2]==7: z=z.flatten(1,2)
            elif z.shape[-2]==7 and z.shape[-1]==7: z=z.flatten(2).transpose(1,2)
            else: z=z.flatten(1,2)
        return z

visual=VisualEncoder().to(DEVICE)
clinical=ClinicalEncoder(layers=2).to(DEVICE)
print("Visual hidden:",visual.hidden," Text hidden:",clinical.hidden)


In [ ]:

# ============================================================
# 7. CBAF
# ============================================================
class CBAF(nn.Module):
    def __init__(self,vdim,tdim,common=128,aff=32,outdim=256):
        super().__init__()
        self.v=nn.Linear(vdim,common); self.t=nn.Linear(tdim,common)
        self.vq=nn.Linear(common,aff,bias=False); self.tk=nn.Linear(common,aff,bias=False)
        self.tq=nn.Linear(common,aff,bias=False); self.vk=nn.Linear(common,aff,bias=False)
        self.tv=nn.Linear(common,common); self.vv=nn.Linear(common,common)
        self.sink_v=nn.Parameter(torch.zeros(1)); self.sink_t=nn.Parameter(torch.zeros(1))
        self.gv=nn.Linear(2*common,common); self.gt=nn.Linear(2*common,common)
        self.ffv=nn.Sequential(nn.LayerNorm(common),nn.Linear(common,2*common),nn.GELU(),nn.Linear(2*common,common))
        self.fft=nn.Sequential(nn.LayerNorm(common),nn.Linear(common,2*common),nn.GELU(),nn.Linear(2*common,common))
        self.pv=nn.Parameter(torch.randn(1,common)); self.pt=nn.Parameter(torch.randn(1,common))
        self.out=nn.Linear(2*common,outdim)

    def sink_softmax(self,s,sink):
        b=sink.expand(s.shape[0],s.shape[1],1)
        return torch.softmax(torch.cat([s,b],-1),-1)[...,:-1]

    def pool(self,x,q):
        a=torch.softmax(torch.einsum("bd,btd->bt",q.expand(x.size(0),-1),x),-1)
        return torch.einsum("bt,btd->bd",a,x)

    def forward(self,vis,txt,mask=None):
        v,t=self.v(vis),self.t(txt)
        a=torch.einsum("bva,bta->bvt",self.vq(v),self.tk(t))/math.sqrt(self.vq.out_features)
        if mask is not None: a=a.masked_fill(mask[:,None,:]==0,-1e4)
        av=self.sink_softmax(a,self.sink_v)
        cv=torch.einsum("bvt,btd->bvd",av,self.tv(t))
        at=self.sink_softmax(a.transpose(1,2),self.sink_t)
        ct=torch.einsum("btv,bvd->btd",at,self.vv(v))
        v=v+torch.sigmoid(self.gv(torch.cat([v,cv],-1)))*cv
        t=t+torch.sigmoid(self.gt(torch.cat([t,ct],-1)))*ct
        v=v+self.ffv(v); t=t+self.fft(t)
        z=self.out(torch.cat([self.pool(v,self.pv),self.pool(t,self.pt)],-1))
        return z,v,t

fusion=CBAF(visual.hidden,clinical.hidden,CBAF_DIM,AFFINITY_DIM,FUSED_DIM).to(DEVICE)
print("CBAF ready. Fused dimension:",FUSED_DIM)


In [ ]:

# ============================================================
# 8. DATASET + MODEL
# ============================================================
MEAN=torch.tensor([0.485,0.456,0.406]).view(3,1,1)
STD=torch.tensor([0.229,0.224,0.225]).view(3,1,1)

def image_tensor(path,augment=False):
    a=preprocess_ultrasound(path,IMAGE_SIZE)
    x=torch.from_numpy(a).float().permute(2,0,1)/255.
    if augment and random.random()<0.5: x=torch.flip(x,[2])
    return (x-MEAN)/STD

class MMSet(Dataset):
    def __init__(self,df,aug=False,mask_prob=0):
        self.df=df.reset_index(drop=True); self.aug=aug; self.mask_prob=mask_prob
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        text="" if random.random()<self.mask_prob else r.masked_narrative
        e=tokenizer(text,padding="max_length",truncation=True,max_length=MAX_TEXT_LEN,return_tensors="pt")
        return {"image":image_tensor(r.image_path,self.aug),
                "input_ids":e.input_ids[0],"attention_mask":e.attention_mask[0],
                "label":torch.tensor(float(r.label))}

class FibroLightVLNet(nn.Module):
    def __init__(self,visual,clinical,fusion,experts=4,width=32):
        super().__init__(); self.visual=visual; self.clinical=clinical; self.fusion=fusion
        self.experts=nn.ModuleList([nn.Sequential(nn.Linear(256,width),nn.GELU(),nn.Dropout(.1),nn.Linear(width,1)) for _ in range(experts)])
        self.gate=nn.Linear(256,experts); self.aux=nn.Linear(256,1); self.n_experts=experts
    def encode(self,img,ids,am):
        v=self.visual(img); t=self.clinical(ids,am); return self.fusion(v,t,am)
    def forward(self,img,ids,am,ret=False):
        z,v,t=self.encode(img,ids,am)
        g=torch.softmax(self.gate(z),-1); e=torch.cat([m(z) for m in self.experts],1)
        logit=(g*e).sum(1); aux=self.aux(z).squeeze(1)
        return (logit,aux,g,z,v,t) if ret else logit

model=FibroLightVLNet(visual,clinical,fusion,4,32).to(DEVICE)
for n,p in model.named_parameters():
    if n.startswith("visual."): p.requires_grad=False
    elif n.startswith("clinical.model.") and ".A" not in n and ".B" not in n: p.requires_grad=False

total=sum(p.numel() for p in model.parameters())
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total/1e6:.2f}M | Trainable: {trainable/1e6:.2f}M")


In [ ]:

# ============================================================
# 9. TRAIN / TEST SPLIT + STAGE 1
# ============================================================
train_df,test_df=train_test_split(data,test_size=.20,stratify=data.label,random_state=SEED)
train_df=train_df.reset_index(drop=True); test_df=test_df.reset_index(drop=True)

train_loader=DataLoader(MMSet(train_df,True,MASK_PROB),batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
train_eval_loader=DataLoader(MMSet(train_df,False,0),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
test_loader=DataLoader(MMSet(test_df,False,0),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)

def focal(logits,y,pos_weight,gamma=2):
    b=F.binary_cross_entropy_with_logits(logits,y,reduction="none",pos_weight=pos_weight)
    p=torch.sigmoid(logits); pt=torch.where(y>0.5,p,1-p)
    return ((1-pt).pow(gamma)*b).mean()

def stage1_train(model):
    model.train()
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    pos=max(float(train_df.label.sum()),1); neg=max(len(train_df)-pos,1)
    pw=torch.tensor([neg/pos],device=DEVICE); hist=[]
    for ep in range(EPOCHS):
        ls=[]
        for b in train_loader:
            img=b["image"].to(DEVICE); ids=b["input_ids"].to(DEVICE); am=b["attention_mask"].to(DEVICE); y=b["label"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logit,aux,g,*_=model(img,ids,am,True)
                loss=focal(logit,y,pw,FOCAL_GAMMA)+.25*focal(aux,y,pw,FOCAL_GAMMA)
                loss += BALANCE_WEIGHT*((g.mean(0)-1/model.n_experts)**2).mean()
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            scaler.step(opt); scaler.update(); ls.append(float(loss.detach().cpu()))
        hist.append(np.mean(ls))
    return hist

history=stage1_train(model)
pd.DataFrame({"epoch":np.arange(1,len(history)+1),"loss":history}).to_csv(f"{OUTPUT_DIR}/stage1_history.csv",index=False)
plt.figure(figsize=(8,5)); plt.plot(range(1,len(history)+1),history,marker="o")
plt.xlabel("Epoch",fontweight="bold"); plt.ylabel("Loss",fontweight="bold")
plt.title("Stage-1 Multimodal Training",fontweight="bold"); plt.tight_layout(); plt.show()


In [ ]:

# ============================================================
# 10. STAGE 2 — FROZEN FUSED EMBEDDINGS
# ============================================================
@torch.no_grad()
def embeddings(model,loader):
    model.eval(); Z=[]; Y=[]; G=[]
    for b in loader:
        out=model(b["image"].to(DEVICE),b["input_ids"].to(DEVICE),b["attention_mask"].to(DEVICE),True)
        Z.append(out[3].cpu().numpy()); Y.append(b["label"].numpy()); G.append(out[2].cpu().numpy())
    return np.vstack(Z),np.concatenate(Y),np.vstack(G)

Ztr,ytr,gtr=embeddings(model,train_eval_loader)
Zte,yte,gte=embeddings(model,test_loader)
print("Train embedding:",Ztr.shape," Test embedding:",Zte.shape)


In [ ]:

# ============================================================
# 11. ADEO — FEATURE MASK + EXPERT SETTINGS
# ============================================================
def prior_score(Z,y):
    a,b=Z[y==1],Z[y==0]
    s=np.abs(a.mean(0)-b.mean(0))/np.sqrt(a.var(0)+b.var(0)+1e-6)
    return s/(s.max()+1e-12)

def decode(g,d):
    m=g[:d]>=.5
    if m.sum()<2: m[np.argsort(g[:d])[-2:]]=True
    e=int(round(EXPERT_MIN+g[d]*(EXPERT_MAX-EXPERT_MIN)))
    w=int(round(WIDTH_MIN+g[d+1]*(WIDTH_MAX-WIDTH_MIN)))
    return m,e,w

def fitness(g,Z,y,prior):
    m,e,w=decode(g,Z.shape[1]); X=Z[:,m]; k=X.shape[1]
    try:
        clf=LogisticRegression(max_iter=250,class_weight="balanced",solver="liblinear").fit(X,y)
        au=average_precision_score(y,clf.predict_proba(X)[:,1])
    except: au=0
    if k>1:
        c=np.corrcoef(X,rowvar=False); c=np.nan_to_num(np.abs(c)); red=(c.sum()-k)/(k*(k-1))
    else: red=0
    comp=(k*w+e*w+e)/134152
    sep=prior[m].mean()
    return .70*(1-au)+.15*red+.15*comp-.05*sep

def adeo(Z,y,pop_size=ADEO_POP,generations=ADEO_GEN):
    rng=np.random.default_rng(SEED); d=Z.shape[1]; prior=prior_score(Z,y)
    P=rng.random((pop_size,d+2)); Fv=np.array([fitness(x,Z,y,prior) for x in P])
    Fm,CRm=.5,.5
    for gen in range(generations):
        order=np.argsort(Fv); pool=order[:max(2,int(.1*pop_size))]
        NP=P.copy(); NF=Fv.copy(); goodF=[]; goodCR=[]; imp=[]
        for i in range(pop_size):
            choices=[j for j in range(pop_size) if j!=i]
            r1,r2=rng.choice(choices,2,False); pb=P[rng.choice(pool)]
            Fi=np.clip(Fm+.1*rng.standard_cauchy(),.1,.9); CRi=np.clip(CRm+.1*rng.normal(),0,1)
            M=P[i]+Fi*(pb-P[i])+Fi*(P[r1]-P[r2])
            M[:d]+=0.05*(prior-.5); M=np.clip(M,0,1)
            jr=rng.integers(d+2)
            T=np.array([M[j] if rng.random()<CRi or j==jr else P[i,j] for j in range(d+2)])
            tf=fitness(T,Z,y,prior)
            if tf<=Fv[i]:
                NP[i]=T; NF[i]=tf; goodF.append(Fi); goodCR.append(CRi); imp.append(Fv[i]-tf)
        P,Fv=NP,NF
        if imp:
            ww=np.array(imp)+1e-8
            Fm=.9*Fm+.1*np.sum(ww*np.array(goodF)**2)/np.sum(ww*np.array(goodF))
            CRm=.9*CRm+.1*np.average(goodCR,weights=ww)
        print(f"ADEO generation {gen+1}/{generations}: best={Fv.min():.6f}")
    g=P[np.argmin(Fv)]; m,e,w=decode(g,d)
    return {"genotype":g,"mask":m,"experts":e,"width":w,"fitness":float(Fv.min()),"prior":prior}

if RUN_ADEO:
    adeo_result=adeo(Ztr,ytr)
else:
    pr=prior_score(Ztr,ytr); m=pr>=np.quantile(pr,.70)
    if m.sum()<2:m[np.argsort(pr)[-2:]]=True
    adeo_result={"mask":m,"experts":4,"width":32,"fitness":np.nan,"prior":pr}

mask=adeo_result["mask"]
print("Selected:",int(mask.sum()),"/",Ztr.shape[1]," Experts:",adeo_result["experts"]," Width:",adeo_result["width"])
np.save(f"{OUTPUT_DIR}/ADEO_mask.npy",mask)


In [ ]:

# ============================================================
# 12. STAGE 4 — LIGHTWEIGHT GATED MoE
# ============================================================
class SelectedMoE(nn.Module):
    def __init__(self,d,e=4,w=32):
        super().__init__()
        self.experts=nn.ModuleList([nn.Sequential(nn.Linear(d,w),nn.GELU(),nn.Dropout(.1),nn.Linear(w,1)) for _ in range(e)])
        self.gate=nn.Linear(d,e); self.e=e
    def forward(self,x):
        g=torch.softmax(self.gate(x),-1)
        z=torch.cat([m(x) for m in self.experts],1)
        return (g*z).sum(1),g

Xtr=torch.tensor(Ztr[:,mask],dtype=torch.float32,device=DEVICE)
Ytr=torch.tensor(ytr,dtype=torch.float32,device=DEVICE)
Xte=torch.tensor(Zte[:,mask],dtype=torch.float32,device=DEVICE)
Yte=torch.tensor(yte,dtype=torch.float32,device=DEVICE)

moe=SelectedMoE(Xtr.shape[1],adeo_result["experts"],adeo_result["width"]).to(DEVICE)
opt=torch.optim.AdamW(moe.parameters(),lr=1e-3,weight_decay=.01)
pw=torch.tensor([max(len(ytr)-ytr.sum(),1)/max(ytr.sum(),1)],device=DEVICE)

for ep in range(12 if FAST_MODE else 50):
    moe.train(); opt.zero_grad()
    logit,g=moe(Xtr)
    loss=focal(logit,Ytr,pw,FOCAL_GAMMA)+BALANCE_WEIGHT*((g.mean(0)-1/moe.e)**2).mean()
    loss.backward(); torch.nn.utils.clip_grad_norm_(moe.parameters(),GRAD_CLIP); opt.step()

moe.eval()
with torch.no_grad(): pte=torch.sigmoid(moe(Xte)[0]).cpu().numpy()

def metrics(y,p,thr=.5):
    pred=(p>=thr).astype(int); cm=confusion_matrix(y,pred,labels=[0,1]); tn,fp,fn,tp=cm.ravel()
    return {
        "Accuracy":accuracy_score(y,pred),
        "Precision":precision_score(y,pred,zero_division=0),
        "Sensitivity":recall_score(y,pred,zero_division=0),
        "Specificity":tn/max(tn+fp,1),
        "F1":f1_score(y,pred,zero_division=0),
        "ROC_AUC":roc_auc_score(y,p),
        "AUPRC":average_precision_score(y,p),
        "Balanced_Accuracy":balanced_accuracy_score(y,pred),
        "MCC":matthews_corrcoef(y,pred)
    }

test_metrics=metrics(yte,pte)
display(pd.DataFrame([test_metrics]))
pd.DataFrame([test_metrics]).to_csv(f"{OUTPUT_DIR}/test_metrics.csv",index=False)

torch.save({"model":model.state_dict(),"moe":moe.state_dict(),"mask":mask,"adeo":adeo_result},
           f"{OUTPUT_DIR}/FibroLight_VLNet_checkpoint.pth")


In [ ]:

# ============================================================
# 13. IMAGE-ONLY / TEXT-ONLY / SHUFFLED-PAIRING CONTROLS
# ============================================================
@torch.no_grad()
def modality_features(model,loader,kind):
    model.eval(); X=[]; Y=[]
    for b in loader:
        img=b["image"].to(DEVICE); ids=b["input_ids"].to(DEVICE); am=b["attention_mask"].to(DEVICE)
        if kind=="image":
            q=model.visual(img).mean(1)
        else:
            q=model.clinical(ids,am); q=(q*am.unsqueeze(-1)).sum(1)/(am.sum(1,keepdim=True)+1e-6)
        X.append(q.cpu().numpy()); Y.append(b["label"].numpy())
    return np.vstack(X),np.concatenate(Y)

def logistic_control(X1,y1,X2,y2):
    c=LogisticRegression(max_iter=500,class_weight="balanced").fit(X1,y1)
    p=c.predict_proba(X2)[:,1]; return metrics(y2,p)

Itr,Iytr=modality_features(model,train_eval_loader,"image")
Ite,Iyte=modality_features(model,test_loader,"image")
Ttr,Tytr=modality_features(model,train_eval_loader,"text")
Tte,Tyte=modality_features(model,test_loader,"text")

rows=[{"Control":"Image-only",**logistic_control(Itr,Iytr,Ite,Iyte)},
      {"Control":"Narrative-only",**logistic_control(Ttr,Tytr,Tte,Tyte)}]

sh=test_df.copy()
sh["masked_narrative"]=np.random.default_rng(SEED).permutation(sh.masked_narrative.values)
sh_loader=DataLoader(MMSet(sh,False,0),batch_size=BATCH_SIZE,shuffle=False)
Zs,ys,_=embeddings(model,sh_loader)
with torch.no_grad(): ps=torch.sigmoid(moe(torch.tensor(Zs[:,mask],dtype=torch.float32,device=DEVICE))[0]).cpu().numpy()
rows.append({"Control":"Shuffled-pairing",**metrics(ys,ps)})
controls=pd.DataFrame(rows); display(controls)
controls.to_csv(f"{OUTPUT_DIR}/leakage_controls.csv",index=False)


In [ ]:

# ============================================================
# 14. NARRATIVE-FREE EXTERNAL SPECIFICITY
# ============================================================
def external_loader(root):
    files=find_images(root)
    return pd.DataFrame({"image_path":files,"label":np.zeros(len(files),dtype=int),
                         "patient_id":[Path(x).stem for x in files],
                         "category":[Path(x).parent.name for x in files],
                         "masked_narrative":[""]*len(files)})

if REAL_DATA_MODE and Path(EXTERNAL_ROOT).exists():
    ext=external_loader(EXTERNAL_ROOT)
else:
    ext=pd.DataFrame()

if len(ext):
    dl=DataLoader(MMSet(ext,False,1),batch_size=BATCH_SIZE,shuffle=False)
    Ze,ye,_=embeddings(model,dl)
    with torch.no_grad(): pe=torch.sigmoid(moe(torch.tensor(Ze[:,mask],dtype=torch.float32,device=DEVICE))[0]).cpu().numpy()
    extm=metrics(ye,pe)
    display(pd.DataFrame([extm]))
    pd.DataFrame([extm]).to_csv(f"{OUTPUT_DIR}/external_specificity.csv",index=False)
else:
    print("External specificity set not configured. Set REAL_DATA_MODE=True and EXTERNAL_ROOT to run it.")



## FMEE explanation

The paper defines a joint player set containing image blocks and narrative clauses, estimates permutation-sampled Shapley values, reports image/narrative reliance, and then performs a sparse attribution-guided counterfactual search. The following implementation keeps the trained detector fixed and provides a practical Colab implementation of that procedure.


In [ ]:

# ============================================================
# 15. FMEE — JOINT MULTIMODAL ATTRIBUTION + COUNTERFACTUAL
# ============================================================
def split_clauses(text):
    return [x.strip() for x in re.split(r"(?<=[.!?])\s+",str(text)) if x.strip()]

def predict_array(gray,text):
    a=np.repeat(gray[...,None],3,axis=2).astype(np.uint8)
    x=torch.from_numpy(a).float().permute(2,0,1)/255.
    x=(x-MEAN)/STD; x=x.unsqueeze(0).to(DEVICE)
    e=tokenizer(text,padding="max_length",truncation=True,max_length=MAX_TEXT_LEN,return_tensors="pt")
    ids=e.input_ids.to(DEVICE); am=e.attention_mask.to(DEVICE)
    with torch.no_grad():
        z,_,_=model.encode(x,ids,am)
        return torch.sigmoid(moe(z[:,mask])[0]).item()

def predict_file(path,text):
    return predict_array(cv2.resize(cv2.imread(path,cv2.IMREAD_GRAYSCALE),(224,224)),text)

def fmee(row,permutations=EXPLAIN_PERMUTATIONS):
    path=row.image_path; text=row.masked_narrative
    raw=cv2.resize(cv2.imread(path,cv2.IMREAD_GRAYSCALE),(224,224))
    clauses=split_clauses(text) or [""]
    base=predict_array(raw,text)
    bg=np.uint8(np.median(raw))

    blocks=[]
    for r in range(7):
        for c in range(7):
            m=np.zeros((224,224),bool); m[r*32:(r+1)*32,c*32:(c+1)*32]=True
            blocks.append(m)

    rng=np.random.default_rng(SEED)
    ia=np.zeros(49); ta=np.zeros(len(clauses))

    for _ in range(permutations):
        cur=np.full_like(raw,bg); curp=predict_array(cur,text)
        for j in rng.permutation(49):
            cur=cur.copy(); cur[blocks[j]]=raw[blocks[j]]
            newp=predict_array(cur,text); ia[j]+=newp-curp; curp=newp

        active=[]; curp=predict_file(path,"")
        for j in rng.permutation(len(clauses)):
            active.append(clauses[j]); newp=predict_file(path," ".join(active))
            ta[j]+=newp-curp; curp=newp

    ia/=max(permutations,1); ta/=max(permutations,1)
    im=np.abs(ia).sum(); tm=np.abs(ta).sum(); den=im+tm+1e-12
    pred=int(base>=.5); oriented=1 if pred else -1
    order_i=np.argsort(-(oriented*ia)); order_t=np.argsort(-(oriented*ta))
    edits=[]; changed=False; current=base

    for j in list(order_i[:8])+list(order_t[:5]):
        if len(edits)>=10: break
        if j<49:
            trial=raw.copy(); trial[blocks[j]]=bg; p2=predict_array(trial,text)
            edits.append({"modality":"image","player":int(j),"probability":float(p2),"delta":float(p2-current)})
        else:
            jj=j-49; kept=[c for k,c in enumerate(clauses) if k!=jj]
            p2=predict_file(path," ".join(kept))
            edits.append({"modality":"narrative","player":int(jj),"probability":float(p2),"delta":float(p2-current)})
        current=p2
        if (p2>=.5)!=pred: changed=True; break

    return {"base_probability":base,"predicted_class":pred,
            "image_reliance":float(im/den),"narrative_reliance":float(tm/den),
            "image_attribution":ia.tolist(),"narrative_attribution":ta.tolist(),
            "clauses":clauses,"counterfactual_success":changed,"counterfactual_edits":edits}

if RUN_EXPLANATIONS:
    ex=fmee(test_df.iloc[0])
    with open(f"{OUTPUT_DIR}/FMEE_example.json","w") as f: json.dump(ex,f,indent=2)
    print("Probability:",ex["base_probability"])
    print("Image reliance:",ex["image_reliance"])
    print("Narrative reliance:",ex["narrative_reliance"])
    print("Counterfactual success:",ex["counterfactual_success"])

    heat=np.array(ex["image_attribution"]).reshape(7,7)
    plt.figure(figsize=(7,6))
    plt.imshow(cv2.resize(cv2.imread(test_df.iloc[0].image_path,0),(224,224)),cmap="gray")
    plt.imshow(cv2.resize(heat,(224,224)),alpha=.45,cmap="coolwarm")
    plt.axis("off"); plt.title("FMEE Image Attribution",fontweight="bold")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/FMEE_image_attribution.png",dpi=300); plt.show()
else:
    ex={}


In [ ]:

# ============================================================
# 16. ROC / PR / CONFUSION MATRIX
# ============================================================
fpr,tpr,_=roc_curve(yte,pte)
plt.figure(figsize=(7,6)); plt.plot(fpr,tpr,label=f"AUC={roc_auc_score(yte,pte):.4f}"); plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate",fontweight="bold"); plt.ylabel("True Positive Rate",fontweight="bold")
plt.title("ROC — FibroLight-VLNet",fontweight="bold"); plt.legend(); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ROC_curve.png",dpi=300); plt.show()

pr,rc,_=precision_recall_curve(yte,pte)
plt.figure(figsize=(7,6)); plt.plot(rc,pr,label=f"AUPRC={average_precision_score(yte,pte):.4f}")
plt.xlabel("Recall",fontweight="bold"); plt.ylabel("Precision",fontweight="bold")
plt.title("Precision–Recall Curve",fontweight="bold"); plt.legend(); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/PR_curve.png",dpi=300); plt.show()

cm=confusion_matrix(yte,(pte>=.5).astype(int))
plt.figure(figsize=(6,5)); sns.heatmap(cm,annot=True,fmt="d",cbar=False)
plt.xlabel("Predicted",fontweight="bold"); plt.ylabel("Actual",fontweight="bold")
plt.title("Confusion Matrix",fontweight="bold"); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png",dpi=300); plt.show()


In [ ]:

# ============================================================
# 17. ABLATION + COMPUTATIONAL PROFILE
# ============================================================
ab=[
    {"Variant":"Full FibroLight-VLNet",**test_metrics},
    {"Variant":"No ADEO / all fused features",**(lambda z: z)(
        metrics(yte,LogisticRegression(max_iter=500,class_weight="balanced").fit(Ztr,ytr).predict_proba(Zte)[:,1])
    )},
    {"Variant":"Image-only",**logistic_control(Itr,Iytr,Ite,Iyte)},
    {"Variant":"Narrative-only",**logistic_control(Ttr,Tytr,Tte,Tyte)}
]
ablation=pd.DataFrame(ab); display(ablation)
ablation.to_csv(f"{OUTPUT_DIR}/ablation_results.csv",index=False)

profile=pd.DataFrame([{
    "Total_parameters":sum(p.numel() for p in model.parameters()),
    "Trainable_parameters":sum(p.numel() for p in model.parameters() if p.requires_grad),
    "Selected_features":int(mask.sum()),
    "Experts":int(adeo_result["experts"]),
    "Expert_width":int(adeo_result["width"]),
    "Paper_visual_GMACs":3.07,
    "Paper_text_GMACs":3.83,
    "Paper_fusion_GMACs":0.07,
    "Paper_total_GMACs":6.97,
    "Paper_INT8_storage_MB":42.0
}])
display(profile)
profile.to_csv(f"{OUTPUT_DIR}/computational_profile.csv",index=False)


In [ ]:

# ============================================================
# 18. SAVE RESULTS TO EXCEL + ZIP
# ============================================================
summary={
    "framework":"FibroLight-VLNet",
    "positive_class":"thecoma-fibroma group / theca-cell-tumor",
    "train_samples":int(len(train_df)),
    "test_samples":int(len(test_df)),
    "selected_features":int(mask.sum()),
    "experts":int(adeo_result["experts"]),
    "expert_width":int(adeo_result["width"]),
    "test_metrics":test_metrics,
    "image_reliance":ex.get("image_reliance"),
    "narrative_reliance":ex.get("narrative_reliance")
}
with open(f"{OUTPUT_DIR}/experiment_summary.json","w") as f: json.dump(summary,f,indent=2)
with open(f"{OUTPUT_DIR}/ADEO_result.json","w") as f:
    json.dump({k:(v.tolist() if isinstance(v,np.ndarray) else v) for k,v in adeo_result.items() if k!="prior"},f,indent=2)

xlsx=f"{OUTPUT_DIR}/FibroLight_VLNet_Results.xlsx"
with pd.ExcelWriter(xlsx,engine="openpyxl") as w:
    pd.DataFrame([test_metrics]).to_excel(w,index=False,sheet_name="Test_Metrics")
    controls.to_excel(w,index=False,sheet_name="Leakage_Controls")
    ablation.to_excel(w,index=False,sheet_name="Ablation")
    profile.to_excel(w,index=False,sheet_name="Complexity")
    pd.DataFrame({"ADEO_Selected":mask.astype(int)}).to_excel(w,index=False,sheet_name="ADEO_Mask")

zipfile=shutil.make_archive(f"{OUTPUT_DIR}/FibroLight_VLNet_Outputs","zip","root_dir",OUTPUT_DIR)
print("OUTPUT:",OUTPUT_DIR)
print("ZIP:",zipfile)



## Real-data checklist

1. Set `DEMO_MODE=False`, `REAL_DATA_MODE=True`.
2. Set `MMOTU_ROOT` to the downloaded MMOTU OTU_2D directory.
3. Set `MMOTU_METADATA` if you have the MMOTU metadata CSV; otherwise the loader attempts to infer labels from directory names.
4. Set `PMC_PATIENTS_FILE` to a CSV containing a narrative/summary column. The loader recognizes common names such as `summary`, `patient_summary`, `text`, and `abstract`.
5. For the external specificity test, set `EXTERNAL_ROOT` to the second-site ovarian ultrasound directory. Because that source has no fibroma-positive class, the notebook reports specificity only.
6. Run all cells from the beginning.
7. Set `FAST_MODE=False` for the paper-scale epoch/ADEO configuration. This is computationally expensive.
8. The notebook intentionally keeps the paper's distinction between **class-conditional pseudo-pairs** and true patient-specific image-text pairs.
